# Simplex by Hand

Original LP:

$$
\begin{aligned}
\max \qquad 7x_1& + 2x_2& & & + 20x_5& &\\

\text{subject to}
\qquad -x_1& + 2x_2& + x_3& - x_4& & &\le 4\\
5x_1& + x_2& & + x_4& + 2x_5& &= 20\\
2x_1& + 2x_2& & & + 15x_5& &\le 10\\
x_1, \dots, x_5 \ge 0
\end{aligned}
$$

Reformulate as minimization: negate objective coeffs

$$
\begin{aligned}
\min \qquad -7x_1& - 2x_2& & & - 20x_5& &\\

\text{subject to}
\qquad -x_1& + 2x_2& + x_3& - x_4& & &\le 4\\
5x_1& + x_2& & + x_4& + 2x_5& &= 20\\
2x_1& + 2x_2& & & + 15x_5& &\le 10\\
x_1, \dots, x_5 \ge 0
\end{aligned}
$$

Introduce slacks: inequalities -> equalities

$$
\begin{aligned}
\min \qquad -7x_1& - 2x_2& & & - 20x_5& & & &\\

\text{subject to}
\qquad -x_1& + 2x_2& + x_3& - x_4& & + s_1& & &= 4\\
5x_1& + x_2& & + x_4& + 2x_5& & & &= 20\\
2x_1& + 2x_2& & & + 15x_5& & + s_2& &= 10\\
x_1, \dots, x_5 \ge 0
\end{aligned}
$$

## Phase 1

Introduce artificals to find initial BFS.

$$
\begin{aligned}
\min \qquad &&&&&&& y_1& &\\

\text{subject to}
\qquad -x_1& + 2x_2& + x_3& - x_4& & + s_1& && &= 4\\
5x_1& + x_2& & + x_4& + 2x_5& & & + y_1& &= 20\\
2x_1& + 2x_2& & & + 15x_5& & + s_2& & &= 10\\
x_1, \dots, x_5 \ge 0
\end{aligned}
$$

Write as simplex tableau.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

def show_simplex_tableau(T: np.ndarray, names: list[str], basic: set[str], precision: int = 4):
    df = pd.DataFrame(T, columns=names)
    rhs_name = names[-1]

    def style_cell(row_idx: int, col_name: str) -> str:
        styles = []
        if row_idx == 0 and col_name != rhs_name:
            styles.append("background-color: #d6ecff;")
        elif col_name in basic:
            styles.append("background-color: #d6ffd6;")
        elif row_idx != 0 and col_name == rhs_name:
            styles.append("background-color: #ffd6d6;")
        return "".join(styles)

    def style_df(data: pd.DataFrame) -> pd.DataFrame:
        out = pd.DataFrame("", index=data.index, columns=data.columns)
        for i in range(data.shape[0]):
            for j, c in enumerate(data.columns):
                out.iat[i, j] = style_cell(i, c)
        return out

    styler = (
        df.style
          .apply(style_df, axis=None)
          .set_table_styles([
              {"selector": "th", "props": [("text-align", "center"), ("padding", "2px 8px")]},
              {"selector": "table", "props": [("border-collapse", "collapse")]},
              {"selector": "td, th", "props": [("border", "1px solid #aaa")]}
          ])
          .format(precision=precision)
    )
    display(styler)

In [2]:
T = np.array([
    [0, 0, 0, 0, 0, 0, 0, 1, 0],
    [-1, 2, 1, -1, 0, 1, 0, 0, 4],
    [5, 1, 0, 1, 2, 0, 0, 1, 20],
    [2, 2, 0, 0, 15, 0, 1, 0, 10]
], dtype=np.float64)

names = ["x1","x2","x3","x4","x5","s1","s2","y1","-z"]
basic = set(["s1","s2","y1"])

show_simplex_tableau(T, names, basic)

,x1,x2,x3,x4,x5,s1,s2,y1,-z
0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000
1,-1.0000,2.0000,1.0000,-1.0000,0.0000,1.0000,0.0000,0.0000,4.0000
2,5.0000,1.0000,0.0000,1.0000,2.0000,0.0000,0.0000,1.0000,20.0000
3,2.0000,2.0000,0.0000,0.0000,15.0000,0.0000,1.0000,0.0000,10.0000


Rewrite objective in terms of non-basic vars.

In [3]:
T[0] -= T[2]
show_simplex_tableau(T, names, basic)

,x1,x2,x3,x4,x5,s1,s2,y1,-z
0,-5.0000,-1.0000,0.0000,-1.0000,-2.0000,0.0000,0.0000,0.0000,-20.0000
1,-1.0000,2.0000,1.0000,-1.0000,0.0000,1.0000,0.0000,0.0000,4.0000
2,5.0000,1.0000,0.0000,1.0000,2.0000,0.0000,0.0000,1.0000,20.0000
3,2.0000,2.0000,0.0000,0.0000,15.0000,0.0000,1.0000,0.0000,10.0000


Entering: $x_1$, negative coeff so introducing $x_1$ decreases objective

Leaving:
* $s_1$: no b/c $x_1$ has negative coeff in that row, $x_1$ would become infeasible
* $y_1$: $x_1$ can be max $\frac{20}{5} = 4$
* $s_2$: $x_1$ can be max $\frac{10}{2} = 5$

$y_1$ row constrains $x_1$ the most, so $y_1$ leaves.

In [4]:
def pivot(T: np.ndarray, row: int, col: int) -> np.ndarray:
    T[row] /= T[row, col]
    coeffs = T[:, col].copy()
    coeffs[row] = 0
    T -= coeffs[:, None] * T[row]
    return T

T = pivot(T, 2, 0)
basic.remove("y1")
basic.add("x1")
show_simplex_tableau(T, names, basic)

,x1,x2,x3,x4,x5,s1,s2,y1,-z
0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000
1,0.0000,2.2000,1.0000,-0.8000,0.4000,1.0000,0.0000,0.2000,8.0000
2,1.0000,0.2000,0.0000,0.2000,0.4000,0.0000,0.0000,0.2000,4.0000
3,0.0000,1.6000,0.0000,-0.4000,14.2000,0.0000,1.0000,-0.4000,2.0000


Since there are no negative coefficients in the objective row, we know that we are at optimality.

Since the objective value is 0, we have set all artificials to 0 and we know that we are at a BFS, namely $x_1 = 4$, $s_1 = 8$, and $s_2 = 2$.

## Phase 2

Now we get rid of the artificials, put back the original objective row, and rewrite the objective in terms of the non-basic variables we got from Phase 1.

In [5]:
# Remove artificials.
T = T[:, np.arange(T.shape[1]) != T.shape[1] - 2]
names.remove("y1")

# Put in original objective
T[0] = [-7, -2, 0, 0, -20, 0, 0, 0]
show_simplex_tableau(T, names, basic)

# Rewrite objective in terms of non-basic.
T[0] += 7 * T[2]
show_simplex_tableau(T, names, basic)

,x1,x2,x3,x4,x5,s1,s2,-z
0,-7.0000,-2.0000,0.0000,0.0000,-20.0000,0.0000,0.0000,0.0000
1,0.0000,2.2000,1.0000,-0.8000,0.4000,1.0000,0.0000,8.0000
2,1.0000,0.2000,0.0000,0.2000,0.4000,0.0000,0.0000,4.0000
3,0.0000,1.6000,0.0000,-0.4000,14.2000,0.0000,1.0000,2.0000


,x1,x2,x3,x4,x5,s1,s2,-z
0,0.0000,-0.6000,0.0000,1.4000,-17.2000,0.0000,0.0000,28.0000
1,0.0000,2.2000,1.0000,-0.8000,0.4000,1.0000,0.0000,8.0000
2,1.0000,0.2000,0.0000,0.2000,0.4000,0.0000,0.0000,4.0000
3,0.0000,1.6000,0.0000,-0.4000,14.2000,0.0000,1.0000,2.0000


$x_5$ enters because it has the most negative objective coefficient.

Leaving:
* $s_1$: $x_5$ can be max $\frac{8}{0.4} = 20$
* $x_1$: $x_5$ can be max $\frac{4}{0.4} = 10$
* $s_2$: $x_5$ can be max $\frac{2}{14.2}$
    
$s_2$ leaves because it constrains $x_5$ the most.

In [6]:
T = pivot(T, 3, 4)
basic.remove("s2")
basic.add("x5")
show_simplex_tableau(T, names, basic)

,x1,x2,x3,x4,x5,s1,s2,-z
0,0.0000,1.3380,0.0000,0.9155,0.0000,0.0000,1.2113,30.4225
1,0.0000,2.1549,1.0000,-0.7887,0.0000,1.0000,-0.0282,7.9437
2,1.0000,0.1549,0.0000,0.2113,0.0000,0.0000,-0.0282,3.9437
3,0.0000,0.1127,0.0000,-0.0282,1.0000,0.0000,0.0704,0.1408


Since there are no negative coeffs in the objective row, we know we are at optimality.

The optimal objective for the original maximization LP is $30.423$ ($-30.423$ for the minimzation LP but we flipped signs) when our decision variables are set to $x_1 = 3.944$, $x_5 = 0.141$, and $x_2 = x_3 = x_4 = 0$.

## Duals

Constraint duals

$y_1 = 0$, $y_3 = 1.211$

$-y_1 + 5y_2 + 2y_3 = 7$

$y_2 = (7 - 2y_3) / 5 = 0.91548$


$y_1 = 0, y_2 = 0.915, y_3 = 1.211$

Constraint tighness

$-x_1 + 2x_2 + x_3 - x_4 \le 4 \qquad$ is loose b/c $y_1 = 0$, also see that the $s_1 = 7.94$ so there is slack in the constraint

$5x_1 + x_2 + x_4 + 2x_5 = 20 \qquad$ is tight b/c $y_2 > 0$, equalities are always tight

$2x_1 + 2x_2 + 15x_5 \le 10 \qquad$ is tight b/c $y_3 > 0$

Relaxing the constraint $2x_1 + 2x_2 + 15x_5 \le 10$ (by increasing $10$) would improve the objective.

In [7]:
show_simplex_tableau(T, names, basic)

,x1,x2,x3,x4,x5,s1,s2,-z
0,0.0000,1.3380,0.0000,0.9155,0.0000,0.0000,1.2113,30.4225
1,0.0000,2.1549,1.0000,-0.7887,0.0000,1.0000,-0.0282,7.9437
2,1.0000,0.1549,0.0000,0.2113,0.0000,0.0000,-0.0282,3.9437
3,0.0000,0.1127,0.0000,-0.0282,1.0000,0.0000,0.0704,0.1408


For a fixed basis $B$, $x_B = B^{-1} b$. If we relaxed the 3rd constraint by $\Delta$, then

$$x_B(\Delta) = B^{-1} (b + \Delta e_3) = x_B(0) + \Delta (B^{-1}e_3)$$

where $e_3 = (0, 0, 1)$.

Since the original column for $s_2$ is $e_3$, the column in the optimal simplex tableau is $B^{-1}e_3$. So then we have that,

$$
\begin{pmatrix} s_1(\Delta)\\ x_1(\Delta)\\ x_5(\Delta) \end{pmatrix} =
\begin{pmatrix} s_1^*\\ x_1^*\\ x_5^* \end{pmatrix} +
\Delta \begin{pmatrix} -0.0282\\ -0.0282\\ 0.0704 \end{pmatrix} \ge 0
$$

Since we need the basic variables to stay feasible
* $s_1$ caps $\Delta$ by $\Delta \le 7.9437 / 0.0282 = 281.691$
* $x_1$ caps $\Delta$ by $\Delta \le 3.9437 / 0.0282 = 139.848$

so the most we can relax constraint 3 is by $\Delta = 139.848$.

Then since $y_3$ is the objective increase we can get per unit of relaxation of the 3rd constraint,
we can increase the objective by $\Delta y_3 = 139.848 (1.22113) = 170.773$.

## Gurobi Check

In [9]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

m = gp.Model("matrix1")

x = m.addMVar(shape=(5,), vtype=GRB.CONTINUOUS, name="x")

c = np.array([7, 2, 0, 0, 20], dtype=np.float64)
m.setObjective(c.T @ x, GRB.MAXIMIZE)

r1 = np.array([-1, 2, 1, -1, 0], dtype=np.float64)
m.addConstr(r1.T @ x <= 4, name="r1")

r2 = np.array([5, 1, 0, 1, 2], dtype=np.float64)
m.addConstr(r2.T @ x == 20, name="r2")

r3 = np.array([2, 2, 0, 0, 15], dtype=np.float64)
m.addConstr(r3.T @ x <= 10, name="r3")

m.optimize()
print(x.X)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Arch Linux")

CPU model: AMD Ryzen 7 7840HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 3 rows, 5 columns and 11 nonzeros (Max)
Model fingerprint: 0xdaa87932
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [2e+00, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [4e+00, 2e+01]

Presolve removed 0 rows and 1 columns
Presolve time: 0.01s
Presolved: 3 rows, 4 columns, 10 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.4000000e+02   3.308167e+01   0.000000e+00      0s
       4    3.0422535e+01   0.000000e+00   0.000000e+00      0s

Solved in 4 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.042253521e+01
[3.94366197 0.         0.         0.         0.14084507]
